# FADEC Digital Twin — Entraînement et validation du modèle

Ce notebook entraîne un réseau de neurones pour classer les états **nominal** et **fuite simulée** à partir de `dataset_moteur.csv`.

La préparation respecte les données utilisées dans le firmware `Core/Src/wifi_mqtt.c` :

| Entrée | Colonne CSV | Normalisation |
| --- | --- | --- |
| Régime | `RPM` | `RPM / 7562.3` |
| Température | `Temperature` | `Temperature / 695.3` |
| Débit carburant | `Fuel` | `Fuel / 100.0` |

Le réseau comporte **3 entrées, deux couches de 8 et 4 neurones, et une sortie sigmoïde**. L'export est réalisé au format TensorFlow Lite Float32, compatible en type et dimensions avec la génération X-CUBE-AI existante.

Le jeu de données provient d'une courte acquisition simulée, avec de nombreux états répétés. La séparation par états distincts évite de partager les doublons exacts entre entraînement et validation, mais ne remplace pas une validation sur des sessions indépendantes.

**Exécution :** ouvrir dans Jupyter ou Google Colab, puis placer `dataset_moteur.csv` à côté du notebook. Dépendances : `tensorflow`, `numpy`, `pandas`, `scikit-learn` et `matplotlib`.

Ce notebook génère **un nouveau modèle expérimental** ; il ne reproduit pas les poids du modèle déjà déployé dans le firmware.

## 1. Chargement des données

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

csv_path = Path("dataset_moteur.csv")
if not csv_path.is_file():
    try:
        from google.colab import files as colab_files
    except ImportError as exc:
        raise FileNotFoundError(
            "dataset_moteur.csv introuvable. Place le CSV dans le dossier de travail du notebook."
        ) from exc
    print("Importer dataset_moteur.csv.")
    uploaded = colab_files.upload()
    if csv_path.name not in uploaded:
        raise FileNotFoundError("Le fichier importé doit être nommé dataset_moteur.csv.")

df = pd.read_csv(csv_path)
print(f"Dataset chargé : {len(df)} observations")
display(df.head())

## 2. Préparation et séparation des ensembles

Les observations possédant exactement les mêmes valeurs `(RPM, Temperature, Fuel)` sont regroupées et affectées à un seul ensemble. Cette méthode limite les doublons entre entraînement, validation et test, sans garantir l'indépendance temporelle.

Les états associés à des étiquettes contradictoires sont signalés ; les données sources ne sont pas modifiées.

In [ ]:
FEATURES = ["RPM", "Temperature", "Fuel"]
TARGET = "Label_Fuite"
required = FEATURES + [TARGET]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Colonnes absentes : {missing}")

for col in required:
    df[col] = pd.to_numeric(df[col], errors="raise")
if df[required].isna().any().any():
    raise ValueError("Valeurs manquantes dans le dataset.")
if set(df[TARGET].unique()) != {0, 1}:
    raise ValueError("Les étiquettes doivent être 0 (normal) ou 1 (fuite).")

print("Lignes :", len(df))
print("Classes :", df[TARGET].value_counts().sort_index().to_dict())
print("États distincts sans le label :", len(df[FEATURES].drop_duplicates()))
if "Timestamp" in df:
    t = pd.to_numeric(df["Timestamp"], errors="coerce")
    print("Durée de la séquence (secondes) :", round(float(t.max()-t.min()), 1))
    print("Transitions entre labels :", int((df[TARGET].diff().fillna(0) != 0).sum()))

# Même triplet d'entrée avec des étiquettes contradictoires.
contradictions = (df.groupby(FEATURES)[TARGET].nunique() > 1)
print("États avec étiquettes contradictoires :", int(contradictions.sum()))
if contradictions.any():
    display(contradictions[contradictions].reset_index().drop(columns=[TARGET]))

# Construction des groupes d'états EXACTEMENT identiques, sans utiliser le label.
df = df.copy()
df["state_group"] = df.groupby(FEATURES, sort=False).ngroup()
group_majority = df.groupby("state_group")[TARGET].agg(lambda s: int(s.mean() >= 0.5))
all_group_ids = group_majority.index.to_numpy()

train_val_ids, test_ids = train_test_split(
    all_group_ids, test_size=0.20, random_state=SEED,
    stratify=group_majority.loc[all_group_ids].to_numpy()
)
train_ids, val_ids = train_test_split(
    train_val_ids, test_size=0.20, random_state=SEED,
    stratify=group_majority.loc[train_val_ids].to_numpy()
)

def select_group_rows(ids):
    return df.loc[df["state_group"].isin(ids)].copy()

train_df = select_group_rows(train_ids)
val_df = select_group_rows(val_ids)
test_df = select_group_rows(test_ids)

# Aucun même état moteur ne doit apparaître dans deux ensembles.
sets = [set(part.state_group) for part in [train_df, val_df, test_df]]
assert not (sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2])

# IMPORTANT : elles DOIVENT correspondre aux divisions écrites dans wifi_mqtt.c.
NORMALIZATION_MAX = np.array([7562.3, 695.3, 100.0], dtype=np.float32)

def get_X_y(part):
    X = part[FEATURES].to_numpy(dtype=np.float32) / NORMALIZATION_MAX
    y = part[TARGET].to_numpy(dtype=np.int32)
    return X, y

X_train, y_train = get_X_y(train_df)
X_val, y_val = get_X_y(val_df)
X_test, y_test = get_X_y(test_df)

for name, part in [("Entraînement", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name:14s} : {len(part):3d} lignes, {part.state_group.nunique():2d} états différents, classes {part[TARGET].value_counts().sort_index().to_dict()}")

print("\nAttention : la séparation par groupes d'états identiques ne garantit pas l'indépendance temporelle entre les observations.")
assert X_train.shape[1] == 3
assert set(y_train) == {0, 1}
assert set(y_val) == {0, 1}
assert set(y_test) == {0, 1}


## 3. Architecture du réseau

Réseau dense à deux couches cachées, de 8 et 4 neurones ReLU, suivi d'une sortie sigmoïde pour la classification binaire.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(3,), name="inputs"),
    tf.keras.layers.Dense(8, activation="relu", name="dense_8"),
    tf.keras.layers.Dense(4, activation="relu", name="dense_4"),
    tf.keras.layers.Dense(1, activation="sigmoid", name="fuite_probability"),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")],
)
model.summary()

## 4. Entraînement

Les poids de classes sont calculés à partir des observations d'entraînement pour tenir compte du déséquilibre du jeu de données. L'arrêt anticipé conserve les meilleurs poids selon la perte de validation.

La perte d'entraînement est pondérée par classe, contrairement à la perte de validation. Leurs valeurs absolues ne sont donc pas directement comparables.

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = {int(c): float(w) for c, w in zip(classes, weights)}
print("Poids des classes :", class_weights)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=18, restore_best_weights=True, verbose=1
)
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=8,
    class_weight=class_weights,
    callbacks=[early_stop],
    shuffle=True,
    verbose=1,
)


## 5. Courbes d'apprentissage

Les courbes correspondent aux métriques calculées à chaque époque, sans lissage. La référence horizontale montre la précision d'une prédiction constante de la classe majoritaire du jeu de validation.

Les variations peuvent être importantes étant donné le faible nombre d'états distincts. L'interprétation nécessite également les résultats du test.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(history.history["accuracy"], label="Entraînement (accuracy)")
axes[0].plot(history.history["val_accuracy"], label="Validation (accuracy)")
val_majority_baseline = max(np.mean(y_val == 0), np.mean(y_val == 1))
axes[0].axhline(val_majority_baseline, color="gray", linestyle="--", label=f"Toujours majorité (val) : {val_majority_baseline:.1%}")
axes[0].set_ylim(-0.02, 1.02)
axes[0].set_title("Précision par époque — courbes brutes")
axes[0].set_xlabel("Époque")
axes[0].set_ylabel("Accuracy")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(history.history["loss"], label="Train loss (pondérée)")
axes[1].plot(history.history["val_loss"], label="Val loss (non pondérée)")
axes[1].set_title("Fonction de coût — niveaux non directement comparables")
axes[1].set_xlabel("Époque")
axes[1].set_ylabel("Binary cross-entropy")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Jeu de validation : {len(y_val)} lignes / {val_df.state_group.nunique()} états distincts.")
print("Le jeu de validation contient peu d'états distincts : interpréter les courbes avec prudence.")


## 6. Évaluation

Les résultats sont présentés à la fois par observation et par **état moteur distinct**. L'évaluation par état évite de compter plusieurs fois une même entrée.

Les états à étiquettes contradictoires sont identifiés ; leur étiquette majoritaire est utilisée pour la comparaison par état. Ces métriques ne constituent pas une validation sur une nouvelle acquisition.

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score

metrics = model.evaluate(X_test, y_test, verbose=0, return_dict=True)
probabilities = model.predict(X_test, verbose=0).reshape(-1)
predicted = (probabilities >= 0.5).astype(np.int32)

print("=== PAR LIGNE (les états fréquents comptent davantage) ===")
print("Résultats Keras :", {k: round(float(v), 4) for k, v in metrics.items()})
print("Baseline toujours 'fuite' :", f"{np.mean(y_test == 1):.1%}")
print("Accuracy :", f"{accuracy_score(y_test, predicted):.1%}")
print("Balanced accuracy :", f"{balanced_accuracy_score(y_test, predicted):.1%}")
print("Précision / rappel classe 'fuite' :", f"{precision_score(y_test, predicted, zero_division=0):.1%}", "/", f"{recall_score(y_test, predicted, zero_division=0):.1%}")
print(classification_report(y_test, predicted, labels=[0, 1], target_names=["Normal", "Fuite"], zero_division=0))

# Prédiction agrégée PAR ÉTAT DISTINCT : chaque état n'est compté qu'une fois.
# Les états dont les étiquettes sont contradictoires sont signalés ; leur label majoritaire est utilisé ici pour la comparaison.
state_eval = test_df[["state_group", TARGET]].copy()
state_eval["prob"] = probabilities
state_eval = state_eval.groupby("state_group").agg(
    true_label=(TARGET, lambda s: int(s.mean() >= 0.5)),
    n_labels=(TARGET, "nunique"),
    prob=("prob", "mean"),
    nb_lignes=("prob", "size"),
)
state_y = state_eval.true_label.to_numpy(dtype=np.int32)
state_pred = (state_eval.prob.to_numpy() >= 0.5).astype(np.int32)
print("\n=== PAR ÉTAT DISTINCT (chaque triplet RPM/Temp/Fuel compté une fois) ===")
print("États test :", len(state_eval), "| étiquettes contradictoires :", int((state_eval.n_labels > 1).sum()))
print("Accuracy par état :", f"{accuracy_score(state_y, state_pred):.1%}")
print("Balanced accuracy par état :", f"{balanced_accuracy_score(state_y, state_pred):.1%}")
print("Baseline majorité par état :", f"{max(np.mean(state_y==0), np.mean(state_y==1)):.1%}")

fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(confusion_matrix(state_y, state_pred, labels=[0, 1]),
                       display_labels=["Normal", "Fuite"]).plot(ax=ax, values_format="d")
ax.set_title("Confusion — états distincts du test")
plt.tight_layout()
plt.show()

print("\nÉvaluation limitée aux états simulés présents dans ce fichier.")


## 7. Export TensorFlow Lite

Le fichier `fadec_edge_ai_model_retrained.tflite` est distinct du modèle `fadec_edge_ai_model.tflite` utilisé actuellement dans le dépôt.

Les dimensions et le format Float32 sont vérifiés après la conversion. Avant un déploiement sur la carte STM32, le nouveau modèle doit être validé, puis réimporté dans X-CUBE-AI pour régénérer le code C.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tflite_model = converter.convert()

out_path = Path("fadec_edge_ai_model_retrained.tflite")
out_path.write_bytes(tflite_model)
print(f"Modèle exporté : {out_path} ({out_path.stat().st_size} octets)")

interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
in_info = interpreter.get_input_details()[0]
out_info = interpreter.get_output_details()[0]
print("Entrée TFLite :", in_info["shape"].tolist(), in_info["dtype"])
print("Sortie TFLite :", out_info["shape"].tolist(), out_info["dtype"])

if in_info["shape"].tolist() != [1, 3] or out_info["shape"].tolist() != [1, 1]:
    raise ValueError("Dimensions TFLite inattendues pour le firmware STM32")
if in_info["dtype"] != np.float32 or out_info["dtype"] != np.float32:
    raise ValueError("Le modèle exporté n'est pas en float32")

print("Modèle exporté dans le dossier de travail. Le modèle embarqué existant n’est pas modifié.")

## Limites et perspectives

- Collecter plusieurs sessions de simulation indépendantes pour les deux classes, à différents régimes.
- Vérifier les étiquettes au voisinage des transitions nominal/fuite.
- Réserver une session complète jamais vue pour l'évaluation finale.
- Compléter l'accuracy avec la balanced accuracy, le rappel et la matrice de confusion.
- Mesurer l'inférence et l'empreinte mémoire sur STM32 avant toute mise à jour du firmware.